# Week 15: Deployment & Fine-tuning

**Lab companion to [week_15.md](../agenda/week_15.md)**

In this lab, you will:
1. Create a REST API for your AI app
2. Prepare data for fine-tuning
3. Fine-tune a model
4. Deploy with Docker

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. Building a REST API

In [ ]:
# Install FastAPI
!pip install fastapi uvicorn -q

In [ ]:
# Create a simple API (run this as a script)
api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
app = FastAPI(title="AI Chat API")

class ChatRequest(BaseModel):
    message: str
    system_prompt: str = "You are a helpful assistant."
    model: str = "gpt-4o-mini"

class ChatResponse(BaseModel):
    response: str
    model: str
    tokens: int

@app.get("/health")
def health_check():
    return {"status": "healthy"}

@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    try:
        response = client.chat.completions.create(
            model=request.model,
            messages=[
                {"role": "system", "content": request.system_prompt},
                {"role": "user", "content": request.message}
            ]
        )
        return ChatResponse(
            response=response.choices[0].message.content,
            model=response.model,
            tokens=response.usage.total_tokens
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("api_server.py", "w") as f:
    f.write(api_code)

print("Created api_server.py")
print("Run with: python api_server.py")
print("API docs at: http://localhost:8000/docs")

In [ ]:
# Test API with requests (when server is running)
import requests

def test_api():
    base_url = "http://localhost:8000"
    
    # Health check
    health = requests.get(f"{base_url}/health")
    print(f"Health: {health.json()}")
    
    # Chat
    chat_response = requests.post(
        f"{base_url}/chat",
        json={"message": "Hello!"}
    )
    print(f"Chat: {chat_response.json()}")

# Uncomment when server is running:
# test_api()

## 2. Preparing Fine-tuning Data

In [ ]:
# Create training data in JSONL format
training_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a customer support agent for TechCo."},
            {"role": "user", "content": "How do I reset my password?"},
            {"role": "assistant", "content": "To reset your password: 1) Go to Settings > Security. 2) Click 'Reset Password'. 3) Follow the email instructions. Need more help?"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a customer support agent for TechCo."},
            {"role": "user", "content": "What's your refund policy?"},
            {"role": "assistant", "content": "We offer 30-day refunds on all purchases. Simply contact support with your order number. Refunds process within 5-7 business days."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a customer support agent for TechCo."},
            {"role": "user", "content": "Is there a mobile app?"},
            {"role": "assistant", "content": "Yes! Our app is available on iOS and Android. Search 'TechCo' in your app store. It includes all features from the web version."}
        ]
    }
]

# Write JSONL file
with open("training_data.jsonl", "w") as f:
    for example in training_examples:
        f.write(json.dumps(example) + "\n")

print(f"Created training_data.jsonl with {len(training_examples)} examples")

In [ ]:
# Validate training data
def validate_training_data(filepath: str) -> dict:
    """Validate JSONL training data."""
    issues = []
    valid_count = 0
    
    with open(filepath, "r") as f:
        for i, line in enumerate(f, 1):
            try:
                data = json.loads(line)
                
                # Check structure
                if "messages" not in data:
                    issues.append(f"Line {i}: Missing 'messages' key")
                    continue
                
                messages = data["messages"]
                
                # Check roles
                if not any(m["role"] == "assistant" for m in messages):
                    issues.append(f"Line {i}: No assistant message")
                    continue
                
                valid_count += 1
                
            except json.JSONDecodeError as e:
                issues.append(f"Line {i}: Invalid JSON - {e}")
    
    return {
        "valid": len(issues) == 0,
        "valid_examples": valid_count,
        "issues": issues
    }

result = validate_training_data("training_data.jsonl")
print(f"Valid: {result['valid']}")
print(f"Valid examples: {result['valid_examples']}")
if result["issues"]:
    print(f"Issues: {result['issues']}")

## 3. Fine-tuning a Model

In [ ]:
# Upload training file
def upload_training_file(filepath: str) -> str:
    with open(filepath, "rb") as f:
        response = client.files.create(
            file=f,
            purpose="fine-tune"
        )
    return response.id

# NOTE: Uncomment to actually upload
# file_id = upload_training_file("training_data.jsonl")
# print(f"Uploaded file: {file_id}")

print("File upload function ready")
print("NOTE: Fine-tuning requires many more examples (50-100 minimum)")

In [ ]:
# Create fine-tuning job
def create_finetune(file_id: str, model: str = "gpt-4o-mini-2024-07-18", suffix: str = "custom") -> str:
    response = client.fine_tuning.jobs.create(
        training_file=file_id,
        model=model,
        suffix=suffix
    )
    return response.id

# Check fine-tuning status
def check_finetune(job_id: str) -> dict:
    job = client.fine_tuning.jobs.retrieve(job_id)
    return {
        "status": job.status,
        "model": job.fine_tuned_model,
        "created_at": job.created_at
    }

# List fine-tuning jobs
def list_finetunes() -> list:
    jobs = client.fine_tuning.jobs.list(limit=5)
    return [
        {"id": j.id, "status": j.status, "model": j.fine_tuned_model}
        for j in jobs.data
    ]

print("Fine-tuning functions ready")
print("\nExample usage:")
print('  job_id = create_finetune(file_id, suffix="techco-support")')
print('  status = check_finetune(job_id)')

In [ ]:
# Using a fine-tuned model
def use_finetuned_model(model_name: str, message: str) -> str:
    """Use a fine-tuned model."""
    response = client.chat.completions.create(
        model=model_name,  # e.g., "ft:gpt-4o-mini-2024-07-18:org::id"
        messages=[
            {"role": "system", "content": "You are a customer support agent for TechCo."},
            {"role": "user", "content": message}
        ]
    )
    return response.choices[0].message.content

print("Usage: use_finetuned_model('ft:gpt-4o-mini:...', 'How do I cancel?')")

## 4. Docker Deployment

In [ ]:
# Create Dockerfile
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app
COPY api_server.py .

# Expose port
EXPOSE 8000

# Run
CMD ["python", "api_server.py"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

# Create requirements.txt
requirements = '''fastapi>=0.100.0
uvicorn>=0.23.0
openai>=1.0.0
python-dotenv>=1.0.0
pydantic>=2.0.0
'''

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("Created Dockerfile and requirements.txt")
print("\nBuild: docker build -t ai-api .")
print("Run: docker run -p 8000:8000 -e OPENAI_API_KEY=sk-... ai-api")

In [ ]:
# Create docker-compose.yml
compose = '''version: '3.8'

services:
  ai-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
'''

with open("docker-compose.yml", "w") as f:
    f.write(compose)

print("Created docker-compose.yml")
print("\nRun: docker-compose up -d")

## Summary

You learned:
- ✅ Building REST APIs with FastAPI
- ✅ Preparing fine-tuning data
- ✅ Fine-tuning models
- ✅ Docker deployment

**Next:** [Week 16 - Capstone & Interview Prep](week_16_capstone.ipynb)